# 🏢 A. Singhmaar Corp — Gemma 4 E2B Fine-Tune **v3**
### Before running:
1. **Runtime → Change runtime type → T4 GPU**
2. **Upload** `singhmaar_corp_finetune_dataset.json` via Files panel (left sidebar 📁)
3. **Run all cells top to bottom** — do not skip any

> ⚠️ Do NOT re-run individual cells — run sequentially. If runtime crashes, use Runtime → Disconnect and delete runtime, then start fresh.

In [ ]:
# ── CELL 1: CONFIG ─────────────────────────────────────────────
import os
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'

HF_TOKEN     = 'YOUR_HUGGINGFACE_TOKEN_HERE'  # Add your HF token here
HF_PUSH_REPO = 'anubhavsinghmaar/singhmaar-gemma4-e2b-ft'
DATASET_FILE = '/content/singhmaar_corp_finetune_dataset.json'

MAX_SEQ_LEN = 512
LORA_RANK   = 16
EPOCHS      = 3
BATCH_SIZE  = 2
GRAD_ACCUM  = 4
LR          = 2e-4

TEST_PROMPTS = [
    'What is the maternity leave policy at A. Singhmaar Corp?',
    'Our enterprise client needs a SOC 2 Type II report for SinghmaarTech. What do I send them?',
    "A tenant's HVAC has been broken for 3 days at SinghmaarProperties. How should I respond?",
    'What authentication methods does the SinghmaarTech API support?',
    "What is SinghmaarCapital's criteria for investing in early-stage startups?",
]

print('✅ Config ready')
print(f'   HF Repo: {HF_PUSH_REPO}')
print(f'   Dataset: {DATASET_FILE}')

In [ ]:
# ── CELL 2: INSTALL (pinned versions) ──────────────────────────
# Pinned to avoid unsloth-zoo version conflicts
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'unsloth[colab-new]', '--quiet', '--upgrade'
], check=True)

# Pin compatible versions as required by unsloth-zoo 2026.4.x
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'trl==0.24.0',
    'datasets==4.3.0',
    'transformers==5.5.0',
    'accelerate',
    'bitsandbytes',
    'sentencepiece',
], check=True)

import torch
print('✅ Deps installed')
print(f'   PyTorch: {torch.__version__}')
print(f'   CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'   GPU: {gpu.name} | {gpu.total_memory/1024**3:.1f} GB VRAM')

In [ ]:
# ── CELL 3: LOAD BASE MODEL ────────────────────────────────────
import os
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'

from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name='unsloth/gemma-4-e2b-it',
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    full_finetuning=False,
    token=HF_TOKEN,
)
print('✅ Gemma-4 E2B loaded')
print(f'   Params: {sum(p.numel() for p in model.parameters())/1e9:.2f}B')

In [ ]:
# ── CELL 4: BASELINE INFERENCE ─────────────────────────────────
from unsloth.chat_templates import get_chat_template
import torch

tokenizer = get_chat_template(tokenizer, chat_template='gemma-4')
FastModel.for_inference(model)

def generate(prompt, context='', max_new_tokens=200):
    user_text = f'{prompt}\n\nContext:\n{context}' if context.strip() else prompt
    # Gemma-4 is multimodal — content must be list of dicts
    messages = [{'role': 'user', 'content': [{'type': 'text', 'text': user_text}]}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True,
        add_generation_prompt=True, return_tensors='pt'
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs, max_new_tokens=max_new_tokens,
            temperature=0.7, do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True).strip()

print('🔵 BASELINE — Gemma-4 E2B (before fine-tuning)')
print('=' * 65)
baseline_results = {}
for prompt in TEST_PROMPTS:
    resp = generate(prompt)
    baseline_results[prompt] = resp
    print(f'\n📌 {prompt}')
    print(f'   {resp[:300]}')
    print('-' * 65)
print('\n✅ Baseline saved')

In [ ]:
# ── CELL 5: LOAD & FORMAT DATASET ──────────────────────────────
# Ensure singhmaar_corp_finetune_dataset.json is in /content/
import json
from datasets import Dataset

with open(DATASET_FILE, 'r', encoding='utf-8') as f:
    raw = json.load(f)
print(f'✅ Loaded {len(raw)} samples | Keys: {list(raw[0].keys())}')

# Alpaca → Gemma-4 multimodal chat format
def to_chat(sample):
    user_content = sample['instruction']
    if sample['input'].strip():
        user_content += f"\n\nContext:\n{sample['input']}"
    return {
        'conversations': [
            {'role': 'user',      'content': [{'type': 'text', 'text': user_content}]},
            {'role': 'assistant', 'content': [{'type': 'text', 'text': sample['output']}]},
        ]
    }

dataset = Dataset.from_list([to_chat(s) for s in raw])

def apply_template(examples):
    return {'text': [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        ).removeprefix(tokenizer.bos_token)
        for convo in examples['conversations']
    ]}

dataset = dataset.map(apply_template, batched=True)
splits  = dataset.train_test_split(test_size=0.1, seed=42)
print(f'✅ Train: {len(splits["train"])} | Eval: {len(splits["test"])}')
print(f'\nSample preview:\n{splits["train"][0]["text"][:300]}...')

In [ ]:
# ── CELL 6: APPLY LoRA ─────────────────────────────────────────
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA applied (rank={LORA_RANK})')
print(f'   Trainable: {trainable:,} ({100*trainable/total:.2f}%)')
print(f'   Frozen:    {total-trainable:,}')

In [ ]:
# ── CELL 7: TRAIN ──────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig
import torch

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=splits['train'],
    eval_dataset=splits['test'],
    args=SFTConfig(
        output_dir='/content/singhmaar-ft-checkpoints',
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        lr_scheduler_type='cosine',
        warmup_steps=5,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        report_to='none',
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LEN,
        packing=True,
        seed=42,
    ),
)

print(f'🏋️ Training...')
print(f'   Epochs: {EPOCHS} | Eff. batch: {BATCH_SIZE*GRAD_ACCUM} | LR: {LR}')
stats = trainer.train()
print(f'\n✅ Training complete!')
print(f'   Steps: {stats.global_step}')
print(f'   Time:  {stats.metrics["train_runtime"]/60:.1f} mins')
print(f'   Loss:  {stats.metrics["train_loss"]:.4f}')

In [ ]:
# ── CELL 8: PUSH TO HF ─────────────────────────────────────────
# Run this IMMEDIATELY after Cell 7 — before RAM fills up
# No GGUF on Colab (causes OOM on T4) — do it locally via Ollama

print(f'🚀 Pushing LoRA adapters to HF: {HF_PUSH_REPO}')
model.push_to_hub(HF_PUSH_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_PUSH_REPO, token=HF_TOKEN)
print(f'\n✅ Model live at:')
print(f'   https://huggingface.co/{HF_PUSH_REPO}')
print(f'\n📌 To run locally on Mac:')
print(f'   ollama run hf.co/{HF_PUSH_REPO}')

In [ ]:
# ── CELL 9: BASELINE vs FINE-TUNED COMPARISON ──────────────────
# Run after Cell 8 push succeeds
FastModel.for_inference(model)

print('\n' + '='*70)
print('  ⚔️   BASELINE  vs  FINE-TUNED — A. Singhmaar Corp AI')
print('='*70)

for i, prompt in enumerate(TEST_PROMPTS, 1):
    ft_resp = generate(prompt)
    print(f'\n[{i}/{len(TEST_PROMPTS)}] 🎯 {prompt}')
    print(f'\n  🔵 BASELINE:\n     {baseline_results[prompt]}')
    print(f'\n  🟢 FINE-TUNED:\n     {ft_resp}')
    print('─'*70)

# Generalization test — not in training data
new_q = "How does SinghmaarHealth handle patient data privacy under India's DPDP Act?"
print(f'\n🆕 GENERALIZATION TEST (unseen prompt):')
print(f'   {new_q}')
print(f'\n  🔵 BASELINE:   {generate(new_q)}')
print(f'\n  🟢 FINE-TUNED: {generate(new_q)}')
print('\n✅ Experiment complete!')